# HTR with Claude

Requires an API key from platform.claude.ai

In [1]:
import anthropic
from collections import Counter
from datetime import datetime
from difflib import SequenceMatcher
from dotenv import load_dotenv
import json
import os
import pandas as pd
from pathlib import Path
import recordlinkage
import regex

from IPython.display import clear_output

In [2]:
def squeal(text=None):
    clear_output(wait=True)
    if not text is None: print(text)

In [3]:
base_directory = "../memories_crawl/scans/bhic/Eindhoven/deel_84"

## 1. Find act-initial text block with Claude

Processing a single image costs about 0.5 cents

In [ ]:
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [ ]:
prompt = """Dear Claude, here is an image displaying two pages with Dutch text 
related to an inheritance. I am interested in the information in the text block 
on the top right of the right page. Could check if that part contains a text like: 
"Memorie van aangifte der nalatenschap van"? If that is the case, can you give me 
the information which follows next? This is 1. the name of the deceased, 2. the 
place of death, and 3. the date of death. Both place and date could be missing. 
If you see a big number next to the text block, that is the act number, which is 
interesting as well. Please return this all information in well-formatted JSON 
format with the keys "act", "name", "place" and "date" and without any comments. 
If the top right of the right page contains a different text or no text at all, 
please return an empty JSON structure."""

In [ ]:
def clear_claude_storage():
    files = client.beta.files.list()
    for file in files:
        client.beta.files.delete(file.id)

In [ ]:
def send_claude_prompt(prompt, upload_file_name):
    try:
        upload_response = client.beta.files.upload(file=Path(upload_file_name))
        message = client.beta.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            messages=[
                {"role": "user", 
                 "content": [
                    {"type": "image",
                     "source": {"type": "file",
                                "file_id": upload_response.id
                               }
                    },
                    {"type": "text", "text": prompt}
                  ]}],
            betas=["files-api-2025-04-14"]
        )
    finally:
        clear_claude_storage()  
    return message.content[0].text

In [ ]:
def add_comment(person_dict, comment_prefix, comment_suffix):
    if comment_prefix:
        if comment_suffix:
            person_dict["comment"] = " ".join([comment_prefix, comment_suffix])
        else:
            person_dict["comment"] = comment_prefix
    elif comment_suffix:
        person_dict["comment"] = comment_suffix

In [ ]:
def add_page_number(person_dict, page_number, sample_file_name):
    sample_file_name_parts = regex.split(r"[_.]", sample_file_name)
    sample_file_name_parts[-2] = str(page_number).zfill(len(sample_file_name_parts[-2]))
    sample_file_name_parts[-2] += "." + sample_file_name_parts[-1]
    sample_file_name_parts.pop()
    person_dict["scan_file"] = "_".join(sample_file_name_parts)
    person_dict["page_number"] = page_number

In [ ]:
def str2dict(string, page_number, sample_file_name):
    groups = regex.search(r"^(.*)```json(.*)```(.*)$", string.strip(), flags=regex.DOTALL)
    person_dict = json.loads(groups.group(2))
    add_comment(person_dict, groups.group(1).strip(), groups.group(1).strip())
    add_page_number(person_dict, page_number, sample_file_name)
    return person_dict

In [ ]:
def save_json(results_json):
    today = datetime.strftime(datetime.now(), "%Y%m%d")
    with open(f"output_{today}.json", "w", encoding='utf-8') as f:
        json.dump(results_json, f, ensure_ascii=False, indent=2)

In [ ]:
def claude2json(results):
    results_json = []
    for page_number, result in results.items():
        results_json.append(str2dict(result, page_number, sample_file_name))
    return results_json

Process scans with Claude. If the block crashes, just run it again to continue

In [ ]:
page_number = 1 if "page_number" not in globals()
starting_pages = [x for x in range(page_number, 469)]
sample_file_name = ""
for file_name in sorted(os.listdir(base_directory)):
    try:
        page_number = int(regex.sub("^0+", "", file_name.split("_")[-1].split('.')[0]))
    except ValueError:
        continue
    if page_number in starting_pages:
        results[page_number] = send_claude_prompt(prompt, os.path.join(base_directory, file_name))
        sample_file_name = file_name
        squeal(page_number)

In [ ]:
results_json = claude2json(results)
save_json(results_json)

## 2. Link HTR information to metadata

In [4]:
def read_json(file_name):
    with open(file_name, "r", encoding='utf-8') as infile:
        return json.load(infile)

In [5]:
def string_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

In [6]:
claude_data = read_json("output_20260521.json")
meta_data = read_json(os.path.join(base_directory, "deeds.json"))

## 2.1 Based on name

Build in name order variation for partial matches

Uses Python package [recordlinkage](https://pypi.org/project/recordlinkage/)

In [7]:
def check_identical_names(meta_name, claude_name):
    claude_name_parts = claude_name.split()
    for split_border in range(0, len(claude_name_parts)):
        claude_name_variant = " ".join(claude_name_parts[split_border:] + claude_name_parts[:split_border])
        if regex.sub(",", "", claude_name_variant.lower()) == meta_name.lower():
            return True
    return False

In [8]:
def compute_name_similarity(meta_name, claude_name):
    claude_name_parts = claude_name.split()
    max_similarity = 0
    for split_border in range(0, len(claude_name_parts)):
        claude_name_variant = " ".join(claude_name_parts[split_border:] + claude_name_parts[:split_border])
        similarity = string_similarity(claude_name_variant.lower(), meta_name.lower())
        if similarity > max_similarity:
            max_similarity = similarity
    return max_similarity

In [9]:
def convert_data_for_recordlinkage(claude_data, meta_data):
    claude_records = [{"name": record["name"],
                       "place": record["place"],
                       "date": record["normalized_date"],
                       "page": record["page_number"]} for record in claude_data if "name" in record]
    claude_records_df = pd.DataFrame(claude_records)
    meta_data_records = [{"name": person["naam_volledig"],
                          "place": person["plaats_overlijden"],
                          "date": person["datum_overlijden"]} for record in meta_data for person in record["personen"]]
    meta_data_records_df = pd.DataFrame(meta_data_records)
    return claude_records_df, meta_data_records_df


def link_with_recordlinkage(claude_data_records_df, meta_data_records_df, threshold=0.85):
    indexer = recordlinkage.Index()
    indexer.full()
    candidate_links = indexer.index(claude_records_df, meta_data_records_df)
    c = recordlinkage.Compare()
    c.string('name', 'name', method='jarowinkler', threshold=threshold)
    c.string('place', 'place', method='jarowinkler', threshold=threshold)
    c.string('date', 'date', method='jarowinkler', threshold=threshold)
    feature_vectors = c.compute(candidate_links, claude_records_df, meta_data_records_df)
    ecm = recordlinkage.ECMClassifier()
    fits = ecm.fit_predict(feature_vectors)
    probs = ecm.prob(feature_vectors)
    return probs[probs > threshold]

In [72]:
def remove_suffix_from_name(name):
    return regex.sub(",* *(in leven|echtgenoot|echtgenote|weduwe|weduwnaar).*$", "", name, regex.IGNORECASE)

In [71]:
def normalize_name(name):
    return " ".join(sorted(remove_suffix_from_name(name).lower().split()))

In [58]:
def merge_matches(matches, matches_normalized):
    claude_ids = matches.index.get_level_values(0)
    claude_ids_normalized = matches_normalized.index.get_level_values(0)
    missing_ids = matches_normalized[~claude_ids_normalized.isin(claude_ids)]
    merged = pd.concat([matches, missing_ids]).sort_index()
    return merged

In [78]:
claude_records_df, meta_data_records_df = convert_data_for_recordlinkage(claude_data, meta_data)
matches = link_with_recordlinkage(claude_records_df, 
                                  meta_data_records_df, 
                                  threshold=0.8)
print(f"number of matches: {len(matches)}")

number of matches: 172


In [79]:
claude_records_df_normalized = claude_records_df.copy()
claude_records_df_normalized["name"] = claude_records_df_normalized["name"].apply(lambda name: 
                                                                                  normalize_name(str(name)))
meta_data_records_df_normalized = meta_data_records_df.copy()
meta_data_records_df_normalized["name"] = meta_data_records_df_normalized["name"].apply(lambda name: 
                                                                                        normalize_name(str(name)))
matches_normalized = link_with_recordlinkage(claude_records_df_normalized, 
                                             meta_data_records_df_normalized, 
                                             threshold=0.8)
print(f"number of matches: {len(matches_normalized)}")

number of matches: 39


In [80]:
matches =  merge_matches(matches, matches_normalized)
print(f"number of matches: {len(matches)}")

number of matches: 181


In [90]:
claude_records_df.to_json("claude_records.json", orient='records')
meta_data_records_df.to_json("meta_data_records.json",  orient='records')

In [81]:
for claude_record_nbr, meta_data_record_nbr in matches.index:
    claude_record = claude_records_df.iloc[claude_record_nbr]
    meta_data_record = meta_data_records_df.iloc[meta_data_record_nbr]
    print(f"{claude_record['page']} {claude_record['name']} {meta_data_record['name']}")

2 Dorothea van Ummelen Dorothea van Ummelen
4 Johanna Maria Davis weduwe van Francis Boon Johanna Maria Daris
5 Wouter van Lieshout Wouter van Lieshout
6 Wouter van Lieshout Wouter van Lieshout
7 Christina de Krom weduwe Johannes Sanders Christina de Krom
17 Wilhelmina Jacobs Wilhelmina Jacobs
19 Maria van Alphen, echtgenoot van Cornelis Johannes Hubertus Spoorenberg Maria van Alphen
21 Martinus Alphons Martinus Johannes Nieuwenhuijsen
22 Judocus Schespers Judocus Scheepers
29 Barthelina Caane Barthelina Caane
30 Barthelina Caane Barthelina Caane
33 Woutrina Verhoeven, echtgenoot van Willem van Vroenhoven Woutrina Verhoeven
35 Woutrina Verdouw Woutrina Verhoeven
37 Hubertus Hertogs Hubertus Hertogs
40 Gerinus Verhoeven Woutrina Verhoeven
40 Gerinus Verhoeven Egidius Verhoeven
48 Martinus Smeelen Martinus Smeelen
48 Martinus Smeelen Martinus Leijtens
50 Margaretha van der Vlier Margaretha van der Velden
51 Marguerite van Velen Margaretha van der Velden
51 Marguerite van Velen Martinus S

In [51]:
matches.index[0][1]

np.int64(17)

In [146]:
expected_meta_id = 0
for claude_id, meta_id in list(matches.sort_index(level=[1, 0]).index):
    if meta_id > expected_meta_id:
        print(f"missing meta ids: {expected_meta_id} - {meta_id - 1 if meta_id > expected_meta_id + 1 else ''} {meta_data_records[expected_meta_id]['name']}")
    expected_meta_id = meta_id + 1

missing meta ids: 8 -  Willen Carel J. Platteel
missing meta ids: 12 -  Peter Hendricus Keeris
missing meta ids: 16 -  Leonardus Peeters
missing meta ids: 19 -  Antonetta Vreijken
missing meta ids: 22 - 23 Maria Gijsbers
missing meta ids: 25 -  Maria Huijbs
missing meta ids: 31 -  Catharina Wilhelmina Hendriks
missing meta ids: 33 -  Petronella Smolders
missing meta ids: 35 -  Hendrina van den Oever
missing meta ids: 37 -  Hester Wijnbergen
missing meta ids: 41 -  Peter Johannes Custers
missing meta ids: 45 -  Anna Maria van Mierlo
missing meta ids: 47 - 48 Petrus Boudewijns
missing meta ids: 53 -  Hendrikus Fops
missing meta ids: 58 - 59 Catharina van Rooij
missing meta ids: 62 -  Helena Verhagen
missing meta ids: 66 -  Adrianus van Oorschot
missing meta ids: 70 - 71 Petrus de Haas
missing meta ids: 77 -  Maria Catharina van Dijk
missing meta ids: 79 -  Bernardus Helsemans
missing meta ids: 81 -  Cornelia Smolders
missing meta ids: 83 - 84 Geertruida Louwers
missing meta ids: 87 -  An

In [148]:
expected_claude_id = 0
for claude_id, meta_id in list(matches.index):
    if claude_id > expected_claude_id:
        print(f"missing claude ids: {expected_claude_id} - {claude_id - 1 if claude_id > expected_claude_id + 1 else ''} {claude_records[expected_claude_id]['name']}")
    expected_claude_id = claude_id + 1

missing claude ids: 5 - 11 Jacobus Vermisse
missing claude ids: 16 - 19 Van Dijk, Maria Catharina
missing claude ids: 22 -  Verhoeven Kontrina
missing claude ids: 24 -  None
missing claude ids: 27 -  Verhoeven Egidius
missing claude ids: 29 - 33 Egidius Verhoeven te Son
missing claude ids: 35 -  van der Velden Margaretha
missing claude ids: 38 -  Hotten Conrad
missing claude ids: 41 -  Adrianus van Oosterhout
missing claude ids: 43 - 45 
missing claude ids: 51 - 52 Peter Sornuldet
missing claude ids: 54 -  Joannes Cornelis den Hoof
missing claude ids: 56 - 58 Jacoba van den Nieuwen van Antonend Louwers van den Voltshom
missing claude ids: 60 - 61 Hzender Sm Divobi Ybed
missing claude ids: 63 - 66 
missing claude ids: 69 -  Koderryks Andries
missing claude ids: 71 - 76 Vantrechout Johanna
missing claude ids: 78 - 85 
missing claude ids: 87 -  Antonella van Ham
missing claude ids: 90 -  van Gers Maria Helena
missing claude ids: 92 -  Mulders Antonius
missing claude ids: 94 - 95 Salomon S

## 2.2 Based on date and name

In [11]:
nbr_of_exact_matches = 0
nbr_of_close_matches = 0
for claude_record in claude_data:
    if ("normalized_date" in claude_record and 
        regex.search(r"^\d\d\d\d-\d\d-\d\d$", str(claude_record["normalized_date"]))):
        max_similarity = 0
        closest_match = ""
        found = False
        for act in meta_data:
            if "personen" in act:
                for person in act["personen"]:
                    if "datum_overlijden" in person:
                        if person["datum_overlijden"] == claude_record["normalized_date"]:
                            found = True
                            similarity = compute_name_similarity(person["naam_volledig"], 
                                                                 claude_record["name"])
                            if similarity > max_similarity:
                                max_similarity = similarity
                                closest_match = person["naam_volledig"]
        if found:
            print(f"found {claude_record["name"]}; closest match({max_similarity:0.1f}): {closest_match}")
            if max_similarity >= 0.9:
                nbr_of_close_matches += 1
print(f"close matches: {nbr_of_close_matches} ({100*nbr_of_close_matches/len(unique_claude_names):0.0f}%)")

found Dorothea van Ummelen; closest match(1.0): Dorothea van Ummelen
found Joanna Fontijen; closest match(0.7): Johanna Dikkens
found Jacoby Wilhelmina, weduwe van Godfried Verstrappen; closest match(0.4): Cornelis Antonius Govers
found Maria van Alphen, echtgenoot van Cornelis Johannes Hubertus Spoorenberg; closest match(0.4): Maria van Alphen
found Judocus Schespers; closest match(0.9): Judocus Scheepers
found Van Dijk, Maria Catharina; closest match(1.0): Maria Catharina van Dijk
found Clara Catharina van Dijck; closest match(0.9): Maria Catharina van Dijk
found Jaune Barthelina; closest match(0.9): Barthelina Caane
found Barthelina Caane; closest match(1.0): Barthelina Caane
found Barthelina Caane; closest match(1.0): Barthelina Caane
found Verhoeven Kontrina; closest match(0.9): Woutrina Verhoeven
found Woutrina Verdouw; closest match(0.8): Woutrina Verhoeven
found Hubertus Hertogs; closest match(0.4): Hendriena Heesterbeek
found Verhoeven Egidius; closest match(1.0): Egidius Verh

In [ ]:
claude_data[23]